<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L13_Few_shot_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L13: Few-shot Learning - Learning from Examples

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 13 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand few-shot learning and in-context learning concepts
- Master example selection strategies for optimal performance
- Learn how to construct effective few-shot prompts
- Implement few-shot learning with varying numbers of examples
- Compare zero-shot vs few-shot performance
- Apply example diversity and similarity strategies
- Optimize few-shot prompts for different tasks

---

## Concept: What is Few-shot Learning?

**Few-shot learning** is the ability of language models to learn new tasks from just a few examples provided in the prompt, without any parameter updates or fine-tuning.

### The Learning Spectrum:

**Zero-shot Learning:**
- No examples provided
- Task described in natural language
- Model relies on pre-training knowledge

**Few-shot Learning:**
- 1-10 examples provided in prompt
- Model learns pattern from examples
- Better performance than zero-shot

**Fine-tuning:**
- Hundreds to thousands of examples
- Model parameters updated
- Best performance but requires training

### Why Few-shot Learning Matters:

1. **No Training Required** - Works immediately with examples
2. **Better Than Zero-shot** - Examples guide the model
3. **Flexible** - Easy to add or change examples
4. **Cost-Effective** - No GPU training needed
5. **Rapid Iteration** - Test different examples instantly

### How In-Context Learning Works:

Large language models can learn patterns from examples in the prompt:

```
Example 1: Input -> Output
Example 2: Input -> Output
Example 3: Input -> Output
Now predict: New Input -> ?
```

The model recognizes the pattern and applies it to new inputs.

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch accelerate openai -q

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Basic Few-shot Learning

**Few-shot learning** provides examples in the prompt to guide the model's behavior.

### The Few-shot Prompt Structure:

```
[Task Description (optional)]

Example 1:
Input: [example input 1]
Output: [example output 1]

Example 2:
Input: [example input 2]
Output: [example output 2]

Example 3:
Input: [example input 3]
Output: [example output 3]

Input: [new input]
Output:
```

### Key Components:

1. **Examples** - Demonstrate the task pattern
2. **Consistency** - Same format for all examples
3. **Clarity** - Clear input/output separation
4. **Relevance** - Examples match the task

---

In [ ]:
# Step 2: Load Language Model for Few-shot Learning

print("Loading Language Model\n")
print("=" * 60)

# Load GPT-2 for few-shot learning demonstrations
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Create text generation pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

print(f"Model loaded: {model_name}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {generator.device}")
print("\nReady for few-shot learning!")

In [ ]:
# Step 3: Zero-shot vs Few-shot Comparison

print("Zero-shot vs Few-shot Learning Comparison\n")
print("=" * 60)

# Task: Sentiment classification
test_input = "The movie was absolutely fantastic!"

# Zero-shot approach
print("\nZERO-SHOT APPROACH")
print("-" * 60)

zero_shot_prompt = f"""Classify the sentiment as positive or negative.

Text: {test_input}
Sentiment:"""

print(f"Prompt:\n{zero_shot_prompt}\n")

zero_shot_result = generator(
    zero_shot_prompt,
    max_new_tokens=5,
    num_return_sequences=1,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)[0]['generated_text']

print(f"Result: {zero_shot_result[len(zero_shot_prompt):].strip()}")

# Few-shot approach
print("\n\nFEW-SHOT APPROACH")
print("-" * 60)

few_shot_prompt = f"""Classify the sentiment as positive or negative.

Text: I loved every minute of it!
Sentiment: positive

Text: This was a complete waste of time.
Sentiment: negative

Text: Best experience I've ever had!
Sentiment: positive

Text: {test_input}
Sentiment:"""

print(f"Prompt:\n{few_shot_prompt}\n")

few_shot_result = generator(
    few_shot_prompt,
    max_new_tokens=5,
    num_return_sequences=1,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)[0]['generated_text']

print(f"Result: {few_shot_result[len(few_shot_prompt):].strip()}")

print("\n" + "=" * 60)
print("\nFew-shot learning provides clearer guidance through examples!")

## Part 2: Number of Examples (N-shot)

**The number of examples** significantly impacts few-shot performance.

### Common Configurations:

- **1-shot** - Single example (minimal guidance)
- **3-shot** - Three examples (good balance)
- **5-shot** - Five examples (strong guidance)
- **10-shot** - Ten examples (maximum context)

### Trade-offs:

**More Examples:**
- Better performance
- Clearer pattern
- More tokens used
- Longer prompts

**Fewer Examples:**
- Faster inference
- Less token cost
- More flexible
- May be less accurate

### Finding the Sweet Spot:

Test different numbers to find optimal performance vs cost balance.

---

In [ ]:
# Step 4: Varying Number of Examples

print("Impact of Number of Examples (N-shot)\n")
print("=" * 60)

# Test input
test_text = "The product quality exceeded my expectations."

# Define examples
examples = [
    ("This is amazing!", "positive"),
    ("Terrible experience.", "negative"),
    ("I'm very satisfied.", "positive"),
    ("Not worth the money.", "negative"),
    ("Highly recommend!", "positive"),
]

# Test with different numbers of examples
for n_shot in [0, 1, 3, 5]:
    print(f"\n{n_shot}-SHOT LEARNING")
    print("-" * 60)
    
    # Build prompt
    if n_shot == 0:
        prompt = f"""Classify sentiment as positive or negative.

Text: {test_text}
Sentiment:"""
    else:
        prompt = "Classify sentiment as positive or negative.\n\n"
        
        # Add examples
        for i in range(n_shot):
            text, sentiment = examples[i]
            prompt += f"Text: {text}\nSentiment: {sentiment}\n\n"
        
        # Add test input
        prompt += f"Text: {test_text}\nSentiment:"
    
    # Generate
    result = generator(
        prompt,
        max_new_tokens=5,
        num_return_sequences=1,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )[0]['generated_text']
    
    prediction = result[len(prompt):].strip().split()[0] if result[len(prompt):].strip() else "[no output]"
    
    print(f"Examples used: {n_shot}")
    print(f"Prompt tokens: ~{len(tokenizer.encode(prompt))}")
    print(f"Prediction: {prediction}")

print("\n" + "=" * 60)
print("\nMore examples generally improve accuracy but increase cost.")

## Part 3: Example Selection Strategies

**Example selection** is crucial for few-shot performance.

### Selection Strategies:

1. **Random Selection**
   - Pick examples randomly
   - Simple but inconsistent
   - Good baseline

2. **Diverse Selection**
   - Cover different patterns
   - Represent all classes equally
   - Better generalization

3. **Similar Selection**
   - Pick examples similar to test input
   - Requires similarity computation
   - Best for specific cases

4. **Difficulty-based Selection**
   - Include challenging examples
   - Help model learn edge cases
   - Improves robustness

5. **Balanced Selection**
   - Equal examples per class
   - Prevents bias
   - Standard practice

### Best Practices:

- **Balance classes** - Equal representation
- **Diverse examples** - Cover different patterns
- **Clear examples** - Unambiguous labels
- **Representative** - Match target distribution

---

In [ ]:
# Step 5: Example Selection Strategies

print("Example Selection Strategies\n")
print("=" * 60)

# Task: Classify customer feedback urgency
test_feedback = "My account has been locked and I can't access my funds."

# Different example sets

# Strategy 1: Balanced selection (equal classes)
balanced_examples = [
    ("The app is running slowly", "low"),
    ("I can't log into my account!", "high"),
    ("When will the new feature be available?", "low"),
    ("My payment failed but I was charged", "high"),
]

# Strategy 2: Diverse selection (different patterns)
diverse_examples = [
    ("Great service, thank you", "low"),
    ("URGENT: Account compromised!", "high"),
    ("How do I change my password?", "medium"),
    ("System error - can't complete transaction", "high"),
]

# Strategy 3: Similar selection (related to test case)
similar_examples = [
    ("Can't access my account", "high"),
    ("Account locked after failed login", "high"),
    ("Unable to log in", "high"),
    ("Account access issue", "high"),
]

strategies = [
    ("Balanced", balanced_examples),
    ("Diverse", diverse_examples),
    ("Similar", similar_examples)
]

for strategy_name, examples in strategies:
    print(f"\n{strategy_name.upper()} SELECTION")
    print("-" * 60)
    
    # Build prompt
    prompt = "Classify urgency as low, medium, or high.\n\n"
    for text, urgency in examples:
        prompt += f"Feedback: {text}\nUrgency: {urgency}\n\n"
    prompt += f"Feedback: {test_feedback}\nUrgency:"
    
    # Generate
    result = generator(
        prompt,
        max_new_tokens=5,
        num_return_sequences=1,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )[0]['generated_text']
    
    prediction = result[len(prompt):].strip().split()[0] if result[len(prompt):].strip() else "[no output]"
    
    print(f"Strategy: {strategy_name}")
    print(f"Prediction: {prediction}")
    print(f"Examples: {len(examples)}")

print("\n" + "=" * 60)
print("\nDifferent selection strategies can lead to different results!")

## Part 4: Prompt Format and Structure

**Prompt formatting** affects how well the model learns from examples.

### Format Elements:

1. **Delimiters**
   - Clear separation between examples
   - Consistent markers (e.g., "---", blank lines)
   - Help model parse structure

2. **Labels**
   - "Input:", "Output:"
   - "Text:", "Label:"
   - "Question:", "Answer:"
   - Choose based on task

3. **Instructions**
   - Optional task description
   - Can improve clarity
   - Keep concise

4. **Order**
   - Example order can matter
   - Try different orderings
   - Recent examples may have more impact

### Format Comparison:

**Format A (Labeled):**
```
Input: text
Output: label
```

**Format B (Natural):**
```
text -> label
```

**Format C (Conversational):**
```
Q: text
A: label
```

---

In [ ]:
# Step 6: Prompt Format Comparison

print("Prompt Format Impact\n")
print("=" * 60)

test_input = "Translate to French: Hello, how are you?"

# Format 1: Labeled format
print("\nFORMAT 1: LABELED")
print("-" * 60)

format1 = """Input: Translate to French: Good morning
Output: Bonjour

Input: Translate to French: Thank you
Output: Merci

Input: Translate to French: Goodbye
Output: Au revoir

Input: Translate to French: Hello, how are you?
Output:"""

print(format1)
result1 = generator(format1, max_new_tokens=10, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nResult: {result1[len(format1):].strip()}")

# Format 2: Arrow format
print("\n\nFORMAT 2: ARROW")
print("-" * 60)

format2 = """Translate to French: Good morning -> Bonjour
Translate to French: Thank you -> Merci
Translate to French: Goodbye -> Au revoir
Translate to French: Hello, how are you? ->"""

print(format2)
result2 = generator(format2, max_new_tokens=10, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nResult: {result2[len(format2):].strip()}")

# Format 3: Q&A format
print("\n\nFORMAT 3: Q&A")
print("-" * 60)

format3 = """Q: Translate to French: Good morning
A: Bonjour

Q: Translate to French: Thank you
A: Merci

Q: Translate to French: Goodbye
A: Au revoir

Q: Translate to French: Hello, how are you?
A:"""

print(format3)
result3 = generator(format3, max_new_tokens=10, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nResult: {result3[len(format3):].strip()}")

print("\n" + "=" * 60)
print("\nDifferent formats work better for different tasks!")

## Part 5: Task-Specific Few-shot Learning

**Different tasks** benefit from different few-shot approaches.

### Task Categories:

1. **Classification**
   - Sentiment analysis
   - Topic categorization
   - Intent detection
   - Balanced examples crucial

2. **Generation**
   - Text completion
   - Creative writing
   - Code generation
   - Style examples important

3. **Transformation**
   - Translation
   - Summarization
   - Paraphrasing
   - Format examples key

4. **Extraction**
   - Named entity recognition
   - Information extraction
   - Key phrase extraction
   - Pattern examples essential

### Task-Specific Tips:

- **Classification**: Balance classes, clear labels
- **Generation**: Show desired style and length
- **Transformation**: Demonstrate format changes
- **Extraction**: Highlight what to extract

---

In [ ]:
# Step 7: Task-Specific Few-shot Examples

print("Task-Specific Few-shot Learning\n")
print("=" * 60)

# Task 1: Classification (Intent Detection)
print("\nTASK 1: INTENT DETECTION (Classification)")
print("-" * 60)

intent_prompt = """Classify the user intent.

User: I want to cancel my subscription
Intent: cancel_subscription

User: How much does the premium plan cost?
Intent: pricing_inquiry

User: My payment didn't go through
Intent: payment_issue

User: Can I upgrade to the business plan?
Intent:"""

print(intent_prompt)
result = generator(intent_prompt, max_new_tokens=10, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nPrediction: {result[len(intent_prompt):].strip()}")

# Task 2: Generation (Product Description)
print("\n\nTASK 2: PRODUCT DESCRIPTION (Generation)")
print("-" * 60)

generation_prompt = """Write a product description.

Product: Wireless Mouse
Description: Ergonomic wireless mouse with precision tracking and long battery life. Perfect for work and gaming.

Product: Coffee Maker
Description: Programmable coffee maker with thermal carafe. Brews 12 cups and keeps coffee hot for hours.

Product: Yoga Mat
Description:"""

print(generation_prompt)
result = generator(generation_prompt, max_new_tokens=30, temperature=0.7, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nGenerated: {result[len(generation_prompt):].strip()}")

# Task 3: Transformation (Formality Conversion)
print("\n\nTASK 3: FORMALITY CONVERSION (Transformation)")
print("-" * 60)

transform_prompt = """Convert informal text to formal.

Informal: Hey, can you send me that file?
Formal: Could you please send me that file?

Informal: Thanks a lot for your help!
Formal: Thank you very much for your assistance.

Informal: I'm gonna be late to the meeting
Formal:"""

print(transform_prompt)
result = generator(transform_prompt, max_new_tokens=20, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nTransformed: {result[len(transform_prompt):].strip()}")

# Task 4: Extraction (Email Extraction)
print("\n\nTASK 4: EMAIL EXTRACTION (Extraction)")
print("-" * 60)

extraction_prompt = """Extract email addresses from text.

Text: Contact John at john@example.com for details.
Email: john@example.com

Text: Send reports to admin@company.org and backup@company.org
Email: admin@company.org, backup@company.org

Text: Reach out to support@service.com if you need help.
Email:"""

print(extraction_prompt)
result = generator(extraction_prompt, max_new_tokens=15, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nExtracted: {result[len(extraction_prompt):].strip()}")

print("\n" + "=" * 60)
print("\nFew-shot learning adapts to diverse task types!")

## Part 6: Optimizing Few-shot Performance

**Optimization strategies** can significantly improve few-shot results.

### Optimization Techniques:

1. **Example Quality**
   - Use clear, unambiguous examples
   - Ensure correct labels
   - Remove noisy examples

2. **Example Diversity**
   - Cover different patterns
   - Include edge cases
   - Represent all classes

3. **Example Order**
   - Try different orderings
   - Recent examples may matter more
   - Test random vs structured order

4. **Prompt Engineering**
   - Add task instructions
   - Use clear formatting
   - Test different templates

5. **Temperature Tuning**
   - Lower for deterministic tasks
   - Higher for creative tasks
   - Test different values

6. **Number of Examples**
   - Start with 3-5 examples
   - Add more if needed
   - Balance cost vs performance

### Iterative Optimization:

1. Start with baseline (3 examples)
2. Test on validation set
3. Adjust examples/format
4. Measure improvement
5. Repeat until satisfied

---

In [ ]:
# Step 8: Few-shot Optimization Demonstration

print("Few-shot Optimization Strategies\n")
print("=" * 60)

# Test cases for sentiment analysis
test_cases = [
    "This product is amazing!",
    "Worst purchase ever.",
    "It's okay, nothing special.",
    "Absolutely love it!",
    "Complete waste of money."
]

true_labels = ["positive", "negative", "neutral", "positive", "negative"]

# Baseline: Simple examples
print("\nBASELINE: Simple Examples")
print("-" * 60)

baseline_examples = [
    ("Good", "positive"),
    ("Bad", "negative"),
    ("Okay", "neutral"),
]

def test_few_shot(examples, test_cases, true_labels, name):
    """Test few-shot performance with given examples"""
    correct = 0
    
    for test_text, true_label in zip(test_cases, true_labels):
        # Build prompt
        prompt = "Classify sentiment as positive, negative, or neutral.\n\n"
        for text, label in examples:
            prompt += f"Text: {text}\nSentiment: {label}\n\n"
        prompt += f"Text: {test_text}\nSentiment:"
        
        # Generate
        result = generator(
            prompt,
            max_new_tokens=5,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id
        )[0]['generated_text']
        
        prediction = result[len(prompt):].strip().split()[0].lower() if result[len(prompt):].strip() else ""
        
        if prediction == true_label:
            correct += 1
    
    accuracy = correct / len(test_cases)
    print(f"{name}: {accuracy:.1%} ({correct}/{len(test_cases)} correct)")
    return accuracy

baseline_acc = test_few_shot(baseline_examples, test_cases, true_labels, "Baseline")

# Optimization 1: More detailed examples
print("\nOPTIMIZATION 1: Detailed Examples")
print("-" * 60)

detailed_examples = [
    ("This is fantastic and exceeded expectations!", "positive"),
    ("Terrible quality, very disappointed.", "negative"),
    ("It works fine, nothing remarkable.", "neutral"),
]

detailed_acc = test_few_shot(detailed_examples, test_cases, true_labels, "Detailed")

# Optimization 2: More examples
print("\nOPTIMIZATION 2: More Examples (5-shot)")
print("-" * 60)

more_examples = [
    ("Absolutely love this product!", "positive"),
    ("Worst experience ever.", "negative"),
    ("It's decent, does the job.", "neutral"),
    ("Highly recommend to everyone!", "positive"),
    ("Complete waste of money.", "negative"),
]

more_acc = test_few_shot(more_examples, test_cases, true_labels, "More Examples")

# Optimization 3: Diverse examples
print("\nOPTIMIZATION 3: Diverse Examples")
print("-" * 60)

diverse_examples = [
    ("Best purchase I've made!", "positive"),
    ("Don't buy this, it's awful.", "negative"),
    ("Average product, nothing special.", "neutral"),
    ("Five stars, amazing quality!", "positive"),
    ("Broke after one use.", "negative"),
]

diverse_acc = test_few_shot(diverse_examples, test_cases, true_labels, "Diverse")

print("\n" + "=" * 60)
print("\nSummary:")
print(f"  Baseline:        {baseline_acc:.1%}")
print(f"  Detailed:        {detailed_acc:.1%}")
print(f"  More Examples:   {more_acc:.1%}")
print(f"  Diverse:         {diverse_acc:.1%}")
print("\nOptimization improves few-shot performance!")

## Part 7: Advanced Few-shot Techniques

**Advanced techniques** can further enhance few-shot learning.

### Advanced Strategies:

1. **Chain-of-Thought (CoT)**
   - Include reasoning steps in examples
   - Model learns to think step-by-step
   - Better for complex tasks

2. **Self-Consistency**
   - Generate multiple outputs
   - Take majority vote
   - Improves reliability

3. **Dynamic Example Selection**
   - Select examples based on test input
   - Use similarity metrics
   - Requires embedding computation

4. **Example Ordering**
   - Order by difficulty
   - Order by similarity
   - Test different arrangements

5. **Instruction Tuning**
   - Add detailed task instructions
   - Explain desired behavior
   - Guide model explicitly

### Chain-of-Thought Example:

```
Question: If John has 3 apples and buys 5 more, how many does he have?
Thought: John starts with 3 apples. He buys 5 more. 3 + 5 = 8.
Answer: 8 apples
```

---

In [ ]:
# Step 9: Chain-of-Thought Few-shot Learning

print("Chain-of-Thought Few-shot Learning\n")
print("=" * 60)

# Standard few-shot
print("\nSTANDARD FEW-SHOT")
print("-" * 60)

standard_prompt = """Solve the math word problem.

Problem: Sarah has 12 cookies and gives 5 to her friend. How many cookies does she have left?
Answer: 7

Problem: A store has 20 shirts and sells 8. How many shirts remain?
Answer: 12

Problem: Tom has 15 dollars and spends 6 dollars. How much money does he have left?
Answer:"""

print(standard_prompt)
result = generator(standard_prompt, max_new_tokens=10, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nResult: {result[len(standard_prompt):].strip()}")

# Chain-of-Thought few-shot
print("\n\nCHAIN-OF-THOUGHT FEW-SHOT")
print("-" * 60)

cot_prompt = """Solve the math word problem step by step.

Problem: Sarah has 12 cookies and gives 5 to her friend. How many cookies does she have left?
Thought: Sarah starts with 12 cookies. She gives away 5. So 12 - 5 = 7.
Answer: 7

Problem: A store has 20 shirts and sells 8. How many shirts remain?
Thought: The store has 20 shirts initially. After selling 8, we calculate 20 - 8 = 12.
Answer: 12

Problem: Tom has 15 dollars and spends 6 dollars. How much money does he have left?
Thought:"""

print(cot_prompt)
result = generator(cot_prompt, max_new_tokens=30, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]['generated_text']
print(f"\nResult: {result[len(cot_prompt):].strip()}")

print("\n" + "=" * 60)
print("\nChain-of-Thought helps model reason through problems!")

## Part 8: Practical Applications and Best Practices

**Real-world applications** of few-shot learning span many domains.

### Common Use Cases:

1. **Customer Support**
   - Ticket classification
   - Response generation
   - Sentiment analysis

2. **Content Creation**
   - Product descriptions
   - Social media posts
   - Email templates

3. **Data Processing**
   - Format conversion
   - Data extraction
   - Text normalization

4. **Analysis**
   - Sentiment analysis
   - Topic classification
   - Intent detection

### Best Practices Summary:

1. **Start Simple**
   - Begin with 3-5 examples
   - Use clear, simple examples
   - Test basic format first

2. **Iterate and Improve**
   - Test on validation data
   - Adjust examples based on errors
   - Try different formats

3. **Balance Trade-offs**
   - Performance vs cost
   - Accuracy vs speed
   - Complexity vs maintainability

4. **Monitor Performance**
   - Track accuracy over time
   - Identify failure patterns
   - Update examples as needed

5. **Know When to Fine-tune**
   - If few-shot isn't accurate enough
   - When you have labeled data
   - For production systems

---

In [ ]:
# Step 10: Real-world Application Example

print("Real-world Application: Customer Support Ticket Routing\n")
print("=" * 60)

# Customer support tickets
tickets = [
    "My credit card payment was declined but I see a pending charge",
    "How do I reset my password? I forgot it",
    "The mobile app crashes when I try to upload photos",
    "I want to upgrade my account to the premium tier",
    "Your service is terrible and I want a refund immediately",
    "When will the new features be available?",
]

# Few-shot prompt for ticket routing
routing_prompt_template = """Route customer support tickets to the appropriate department.

Ticket: I was charged twice for the same order
Department: billing
Priority: high

Ticket: How do I change my email address?
Department: account
Priority: low

Ticket: The website is down and I can't access my account
Department: technical
Priority: high

Ticket: I'd like to purchase additional licenses
Department: sales
Priority: medium

Ticket: {ticket}
Department:"""

print("Processing customer support tickets...\n")
print("-" * 60)

for i, ticket in enumerate(tickets, 1):
    prompt = routing_prompt_template.format(ticket=ticket)
    
    result = generator(
        prompt,
        max_new_tokens=10,
        temperature=0.3,
        pad_token_id=tokenizer.eos_token_id
    )[0]['generated_text']
    
    # Extract department and priority
    output = result[len(prompt):].strip()
    department = output.split('\n')[0].strip() if output else "unknown"
    
    print(f"Ticket {i}:")
    print(f"  Text: {ticket[:60]}..." if len(ticket) > 60 else f"  Text: {ticket}")
    print(f"  Routed to: {department}")
    print()

print("=" * 60)
print("\nFew-shot learning enables rapid deployment without training!")
print("\nBenefits:")
print("  - No labeled training data needed")
print("  - Easy to update routing rules")
print("  - Quick to deploy and iterate")
print("  - Flexible for changing categories")

## Exercises

### Exercise 1: N-shot Comparison
Compare performance across different numbers of examples:
```python
# Choose a classification task (sentiment, topic, intent, etc.)
# Create a test set of 10+ examples
# Test with 0, 1, 3, 5, and 10 examples
# Calculate accuracy for each
# Plot results to find optimal N
# Analyze cost vs performance trade-off
```

### Exercise 2: Example Selection Optimization
Optimize example selection for a specific task:
```python
# Define a classification task
# Create a pool of 20+ potential examples
# Test different selection strategies:
#   - Random selection
#   - Balanced selection
#   - Diverse selection
#   - Difficulty-based selection
# Measure accuracy for each strategy
# Identify the best approach
```

### Exercise 3: Prompt Format Experimentation
Test different prompt formats:
```python
# Choose a task (translation, summarization, etc.)
# Create 5 different prompt formats:
#   - Labeled (Input:/Output:)
#   - Arrow (input -> output)
#   - Q&A format
#   - Natural language
#   - Custom format
# Test each format on same examples
# Compare quality and consistency
```

### Exercise 4: Chain-of-Thought Implementation
Implement chain-of-thought reasoning:
```python
# Choose a reasoning task (math, logic, analysis)
# Create examples with reasoning steps
# Compare standard vs CoT few-shot
# Measure accuracy improvement
# Analyze when CoT helps most
```

### Exercise 5: Real-world Application
Build a complete few-shot application:
```python
# Choose a real-world use case:
#   - Email classification
#   - Product categorization
#   - Content moderation
#   - Intent detection
# Create 5-10 high-quality examples
# Build few-shot prompt
# Test on 20+ real examples
# Calculate metrics (accuracy, F1, etc.)
# Iterate to improve performance
```

### Exercise 6: Few-shot vs Fine-tuning Analysis
Compare few-shot learning with other approaches:
```python
# Choose a task with available labeled data
# Implement three approaches:
#   1. Zero-shot (no examples)
#   2. Few-shot (5 examples)
#   3. Fine-tuned model (if possible)
# Compare:
#   - Accuracy
#   - Development time
#   - Inference cost
#   - Flexibility
# Document trade-offs and recommendations
```

### Exercise 7: Dynamic Example Selection
Implement similarity-based example selection:
```python
# Create a pool of labeled examples
# For each test input:
#   - Compute similarity to all examples
#   - Select top-k most similar examples
#   - Build few-shot prompt
#   - Generate prediction
# Compare with fixed example selection
# Analyze when dynamic selection helps
```

---

## Key Takeaways

1. **Few-shot learning** enables learning from examples without training
2. **In-context learning** allows models to recognize patterns in prompts
3. **Number of examples** affects performance (typically 3-5 is optimal)
4. **Example selection** is crucial for good performance
5. **Balanced examples** prevent bias toward certain classes
6. **Prompt format** impacts how well models learn
7. **Chain-of-Thought** improves reasoning tasks
8. **Diverse examples** lead to better generalization
9. **Optimization** through iteration improves results
10. **Trade-offs** exist between cost, speed, and accuracy

### Quick Reference:

**Basic Few-shot Prompt:**
```python
prompt = """
Task description (optional)

Input: example 1 input
Output: example 1 output

Input: example 2 input
Output: example 2 output

Input: example 3 input
Output: example 3 output

Input: new input
Output:
"""
```

**Chain-of-Thought Prompt:**
```python
prompt = """
Problem: example problem
Thought: reasoning steps
Answer: solution

Problem: new problem
Thought:
"""
```

**Example Selection Guidelines:**
- Use 3-5 examples as starting point
- Balance classes equally
- Include diverse patterns
- Use clear, unambiguous examples
- Test different selections

**Optimization Checklist:**
- [ ] Clear, consistent formatting
- [ ] Balanced class representation
- [ ] Diverse example patterns
- [ ] Appropriate number of examples
- [ ] Tested on validation set
- [ ] Iterated based on errors

### When to Use Few-shot:

- **Use few-shot when:**
  - You have 5-20 labeled examples
  - Need better accuracy than zero-shot
  - Can't afford fine-tuning
  - Task changes frequently
  - Want rapid prototyping

- **Use zero-shot when:**
  - No examples available
  - Task is very simple
  - Cost is primary concern
  - Maximum flexibility needed

- **Use fine-tuning when:**
  - Have 100+ labeled examples
  - Need maximum accuracy
  - Production deployment
  - Task is stable
  - Can invest in training

---

## Additional Resources

### Papers
- **"Language Models are Few-Shot Learners"** (Brown et al., 2020) - GPT-3 paper introducing few-shot learning
- **"What Makes Good In-Context Examples for GPT-3?"** (Liu et al., 2021) - Example selection strategies
- **"Chain-of-Thought Prompting Elicits Reasoning"** (Wei et al., 2022) - CoT technique
- **"Rethinking the Role of Demonstrations"** (Min et al., 2022) - Analysis of few-shot learning

### Documentation
- [OpenAI Few-shot Learning Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [HuggingFace Prompting Guide](https://huggingface.co/docs/transformers/tasks/prompting)
- [Prompt Engineering Guide](https://www.promptingguide.ai/techniques/fewshot)

### Blog Posts
- [Few-shot Learning with Language Models](https://ai.googleblog.com/2021/10/few-shot-learning-with-language-models.html)
- [In-Context Learning Explained](https://www.anthropic.com/index/in-context-learning)
- [Prompt Engineering Best Practices](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering)

### Interactive Tools
- [GPT-3 Playground](https://platform.openai.com/playground) - Test few-shot prompts
- [PromptBase](https://promptbase.com/) - Prompt marketplace
- [LangChain](https://python.langchain.com/) - Framework for prompt engineering

### Advanced Topics
- **Dynamic Example Selection** - Choosing examples based on test input
- **Self-Consistency** - Multiple generations with voting
- **Instruction Tuning** - Models trained to follow instructions
- **Retrieval-Augmented Generation** - Combining retrieval with generation

---

## Next Lesson

**L14: Model Evaluation Metrics** - Learn how to measure LLM performance using perplexity, BLEU, ROUGE, F1 scores, and other evaluation metrics for comprehensive model assessment.

---

**Congratulations! You now understand few-shot learning!**

*You've learned how to leverage in-context learning, select optimal examples, and build effective few-shot prompts for various tasks without any training.*